In [ ]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from copy import copy
from skimage.filters import gaussian
import xarray as xr

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

In [ ]:
data_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

In [ ]:
image_size = 30
R, G, B, wavelength = S_F.getpixelefficiency()
masks = MSF.MF.get_masks(MSF.mosaic_unit, image_size, image_size)
masks_3d = np.dstack([masks["R"], masks["G"], masks["B"]])
wavelength = wavelength
pixel_QYs = np.vstack([R, G, B])
camera_parameters = {}
camera_parameters["gain"] = np.full((image_size, image_size), np.median(gain))
camera_parameters["offset"] = np.full((image_size, image_size), np.median(offset))
camera_parameters["variance"] = np.full((image_size, image_size), np.median(variance))
camera_parameters["readnoise"] = np.full((image_size, image_size), np.median(readnoise))
camera_parameters["rqe"] = np.full((image_size, image_size), np.median(rqe))
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ['R', 'G', 'B']
camera_parameters["pixel_order_indices"] = [0, 1, 2]
camera_parameters["masks"] = masks_3d

In [ ]:
n_photon_space = np.linspace(500, 20000, 500)
n_bootstrap = 1000
background_photons = 0
pixel_size = 69
NA = 1.49

In [ ]:
import types
fitter = types.SimpleNamespace()
fitter.fit_function = I_AF.WLS_fit_colour
fitter.default_params = np.array(["xc", "yc", "A", "sigma", "b", "R", "G", "B"])

In [ ]:
dyes = ['ATTO 488', 'ATTO 532', 'ATTO 565', 'ATTO 647N', 'ATTO 700', 'ATTO 740']
dyes = dyes[::-1]

In [ ]:
filters = []

In [ ]:
save_folder = '/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241213/data'

In [ ]:
types = ['Smoothed', 'Raw']
for error_type in types:
    fitter.error_type = error_type
    for dye in dyes:
        MSF.test_single_dye_fit_method(
                    dye,
                    filters,
                    wavelength,
                    camera_parameters,
                    fitter,
                    save_folder,
                    n_photon_space,
                    n_bootstrap=n_bootstrap,
                    background_photons=background_photons,
                    NA=NA,
                    pixel_size=pixel_size,
                )